# The five mitotic phases. A 3D visualization. 

#### Figure 2 of the paper

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from mitosim.phases import anaphase, metaphase, prometaphase, prophase, telophase
from mitosim.phases.base import PhaseParams

OUT_DIR = Path("../results/five_phases_figure")

## Build one representative cell per phase

Default `PhaseParams` for every phase except anaphase, which uses
`progress=0.6` (a representative mid/late separation -- see
`PhaseParams.progress`'s own docstring for why this isn't the shared
default) -- these are meant to read as typical mid-course snapshots of
each phase, not edge cases. Fixed seeds keep the figure reproducible.

In [ ]:
PHASE_SEEDS = {"prophase": 1001, "prometaphase": 1002, "metaphase": 1003, "anaphase": 1004, "telophase": 1005}
PHASE_MODULES = {
    "prophase": prophase, "prometaphase": prometaphase, "metaphase": metaphase,
    "anaphase": anaphase, "telophase": telophase,
}
PHASE_PARAMS = {name: PhaseParams() for name in PHASE_SEEDS}
PHASE_PARAMS["anaphase"] = PhaseParams(progress=0.6)

states = {}
for name, seed in PHASE_SEEDS.items():
    rng = np.random.default_rng(seed)
    states[name] = PHASE_MODULES[name].configure(rng, PHASE_PARAMS[name])
    print(f"{name:14s} {len(states[name].chromatids)} chromatids, cell_radius {states[name].cell_radius:.2f} um")

## Rotation-minimizing frame + tube mesh (reused from chromatid_model_figure.ipynb)

Same double-reflection method and hemispherical capsule-style end caps as
the single-chromatid methods figure -- see that notebook for why (avoids
Frenet-frame flips on near-straight segments; avoids both the open-hollow-
end and flat-disk-cap artifacts a naive end cap produces now that
chromatid width is uniform, not tapered, right up to the telomere).
`n_theta`/`n_cap` are coarser here (24/6 vs 160/24) -- appropriate for 92
overlapping chromatids per panel, where individual-facet smoothness on any
one tube is far less perceptible than it was for that figure's single
zoomed-in close-up.

In [3]:
def rotation_minimizing_frames(points: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    n = len(points)
    tangents = np.zeros_like(points)
    tangents[:-1] = points[1:] - points[:-1]
    tangents[-1] = tangents[-2]
    tangents /= np.linalg.norm(tangents, axis=1, keepdims=True)

    arbitrary = np.array([0.0, 0.0, 1.0]) if abs(tangents[0, 2]) < 0.9 else np.array([1.0, 0.0, 0.0])
    normal0 = np.cross(tangents[0], arbitrary)
    normal0 /= np.linalg.norm(normal0)

    normals = np.zeros_like(points)
    normals[0] = normal0
    for i in range(n - 1):
        v1 = points[i + 1] - points[i]
        c1 = np.dot(v1, v1)
        if c1 < 1e-12:
            normals[i + 1] = normals[i]
            continue
        r_l = normals[i] - (2.0 / c1) * np.dot(v1, normals[i]) * v1
        t_l = tangents[i] - (2.0 / c1) * np.dot(v1, tangents[i]) * v1
        v2 = tangents[i + 1] - t_l
        c2 = np.dot(v2, v2)
        normals[i + 1] = r_l if c2 < 1e-12 else r_l - (2.0 / c2) * np.dot(v2, r_l) * v2
    binormals = np.cross(tangents, normals)
    return tangents, normals, binormals


def tube_mesh(centreline: np.ndarray, radii: np.ndarray, n_theta: int = 24, n_cap: int = 6):
    tangents, normals, binormals = rotation_minimizing_frames(centreline)
    theta = np.linspace(0, 2 * np.pi, n_theta, endpoint=False)
    ring_dirs = np.cos(theta)[None, :, None] * normals[:, None, :] + np.sin(theta)[None, :, None] * binormals[:, None, :]
    surface = centreline[:, None, :] + radii[:, None, None] * ring_dirs
    surface = np.concatenate([surface, surface[:, :1, :]], axis=1)

    def hemisphere_cap(center, radius, tangent, normal, binormal, outward_sign):
        s = np.linspace(0, np.pi / 2, n_cap + 1)[1:]
        cap_r = radius * np.cos(s)
        cap_h = radius * np.sin(s) * outward_sign
        rings = (
            center[None, None, :]
            + cap_h[:, None, None] * tangent[None, None, :]
            + cap_r[:, None, None] * (np.cos(theta)[None, :, None] * normal[None, None, :]
                                       + np.sin(theta)[None, :, None] * binormal[None, None, :])[0]
        )
        return np.concatenate([rings, rings[:, :1, :]], axis=1)

    cap_start = hemisphere_cap(centreline[0], radii[0], tangents[0], normals[0], binormals[0], -1.0)[::-1]
    cap_end = hemisphere_cap(centreline[-1], radii[-1], tangents[-1], normals[-1], binormals[-1], 1.0)
    surface = np.concatenate([cap_start, surface, cap_end], axis=0)
    return surface[..., 0], surface[..., 1], surface[..., 2]

## Spindle fibers (centromere-to-pole lines)

A thin line from every chromatid's centromere to its assigned pole, so the
spindle attachment -- not just the chromatid geometry and bare pole
markers -- is actually visible. Pole assignment is a fixed,
rotation-invariant convention (not a real kinetochore-attachment model,
which this simulator doesn't represent): with exactly two poles (true for
all five of these phase generators), sister 0 always pairs with pole
index 1 and sister 1 with pole index 0.

In [4]:
def pole_index_for_chromatid(c, poles: np.ndarray) -> int:
    if len(poles) == 2:
        return 1 if c.sister == 0 else 0
    return int(np.argmin(np.linalg.norm(poles - c.centromere, axis=1)))

## Render

One 3D panel per phase, each chromatid drawn as its real swept tube
(world-frame `Chromatid.points`/`.radii` -- no extra placement needed,
unlike the canonical-local-frame single-chromatid figure), spindle poles
marked in red with their fiber lines. Unlike the single-chromatid figure,
`computed_zorder` is left at its default (`True`) here: that figure forced
it off because its control points are a deliberate annotation layer, drawn
"inside" one opaque tube on purpose. Here the fiber lines are meant to
read as genuinely part of the 3D structure -- weaving behind near
chromatids and in front of far ones -- so they need real depth sorting,
not a fixed draw order. The tradeoff: in telophase especially, where the
poles end up well inside each decondensed mass (see `telophase.py`), true
depth sorting can mean the poles/fibers are mostly or entirely occluded
from a given viewing angle. That's the physically honest outcome, not a
bug -- an earlier version of this figure forced `computed_zorder = False`
specifically to keep them always visible, but that made every fiber line
draw flat on top of every tube regardless of which was actually nearer the
camera, which reads as a pasted-on decal rather than real 3D structure.
Prophase has no poles at all (`np.zeros((0, 3))` -- `prophase.py`:
envelope still intact, no spindle yet), so it correctly shows neither
fibers nor pole markers.

In [ ]:
fig = plt.figure(figsize=(12, 8))
for i, name in enumerate(PHASE_SEEDS):
    state = states[name]
    ax = fig.add_subplot(2, 3, i + 1, projection="3d")

    for c in state.chromatids:
        X, Y, Z = tube_mesh(c.points, c.radii)
        ax.plot_surface(X, Y, Z, color="#ffd5d5", alpha=1.0, linewidth=0, antialiased=True, shade=True)

    poles = state.poles
    if len(poles):
        for c in state.chromatids:
            pole = poles[pole_index_for_chromatid(c, poles)]
            ax.plot([c.centromere[0], pole[0]], [c.centromere[1], pole[1]], [c.centromere[2], pole[2]],
                    color="#d94f4f", alpha=0.55, linewidth=0.9)
        ax.scatter(poles[:, 0], poles[:, 1], poles[:, 2], color="#d94f4f", s=40, depthshade=False)

    # Equal scale on all 3 axes, same reasoning as chromatid_model_figure.ipynb
    # (a non-uniform box_aspect/axis-limit combination distorts each tube's
    # true circular cross-section into an ellipse) -- state.cell_radius is
    # already the model's own conservative bounding-sphere radius, so
    # it's used directly as the shared half-range instead
    # of recomputing one from the mesh points.
    r = state.cell_radius
    ax.set_xlim(-r, r); ax.set_ylim(-r, r); ax.set_zlim(-r, r)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(name, y=0.85)
    ax.title.set_size(20)
    ax.view_init(elev=12, azim=-55)
    ax.axis('off')
    ax.patch.set_alpha(0.0)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.4, wspace=-0.7)

OUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_DIR / "five_phases_figure.pdf", bbox_inches="tight")
fig.savefig(OUT_DIR / "five_phases_figure.png", dpi=300, bbox_inches="tight")
print(f"wrote {OUT_DIR / 'five_phases_figure.pdf'} and .png")
plt.show()